# WP2 — Alignment, Phylogeny & ASR

MAFFT alignment → IQ-TREE tree (ModelFinder + UFBoot) → ancestral reconstruction
(IQ-TREE `--ancestral`) → candidate variant pool with sitewise uncertainty.
Requires `mafft` and `iqtree2` on PATH (see `environment.yml`). See `enzymes.docx` §7.

In [1]:
# Make the src/ package importable when running from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from asr_poc.config import load_config
from asr_poc.io_utils import set_seed
cfg = load_config(ROOT / "config" / "target.yaml")
set_seed(cfg.run.seed)
cfg.paths.ensure_dirs()
print("Target family:", cfg.target.name)

Target family: lipase


## 1. Multiple sequence alignment (MAFFT)

In [2]:
from asr_poc import phylo
msa = phylo.align(cfg)
print("MSA:", msa)

09:40:20 [info     ] run                            cmd=C:\Users\gcv\Downloads\mafft-7.526-win64-signed\mafft-win\mafft.BAT --auto C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\curated_sequences\curated.fasta stage=wp2.phylo
09:41:43 [info     ] aligned                        msa=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\alignments\msa.fasta n=315 stage=wp2.phylo
MSA: C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\alignments\msa.fasta


## 2. Phylogenetic tree (IQ-TREE 2)

In [3]:
tree = phylo.build_tree(cfg, msa)
print("Tree:", tree)

09:41:43 [info     ] run                            cmd=C:\Users\gcv\Downloads\iqtree-3.1.2-Windows\iqtree-3.1.2-Windows\bin\iqtree3.EXE -s C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\alignments\msa.fasta -m LG+I+G -redo -pre C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\phylogeny\iqtree -nt AUTO -fast stage=wp2.phylo
09:43:29 [info     ] tree_built                     stage=wp2.phylo tree=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\phylogeny\tree.treefile
Tree: C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\phylogeny\tree.treefile


## 3. Ancestral sequence reconstruction

In [4]:
state_file = phylo.reconstruct_ancestors(cfg, msa)
print("State file:", state_file)

09:43:29 [info     ] asr_bypass_existing            msg=Using existing ASR state file. stage=wp2.phylo state=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\ancestral_sequences\asr.state
State file: C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\ancestral_sequences\asr.state


## 4. Build candidate pool + sitewise uncertainty

In [5]:
summary = phylo.build_candidate_pool(cfg, state_file=state_file)
summary

09:43:29 [info     ] wp2_complete                   alternates=0 ancestors=2 candidates=2 median_length=322 stage=wp2.phylo


{'ancestors': 2, 'alternates': 0, 'candidates': 2, 'median_length': 322}

In [6]:
import pandas as pd
unc = pd.read_csv(cfg.paths.uncertainty_csv)
unc.head()

,node,site,max_pp,entropy
0,Node1,1,0.9,0.598165
1,Node1,2,0.9,0.598165
2,Node1,3,0.9,0.598165
3,Node1,4,0.9,0.598165
4,Node1,5,0.9,0.598165
